In [ ]:
import json
import re

INPUT_FILE = "swayam_dataset_ext.json"
OUTPUT_FILE = "swayam_phi3_clean.jsonl"

pattern = re.compile(
    r"<s>\[INST\](.*?)\[/INST\](.*?)</s>",
    re.DOTALL
)

kept = 0
skipped = 0

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
    for item in data:
        text = item.get("text", "")
        match = pattern.search(text)

        if not match:
            skipped += 1
            continue

        user = match.group(1).strip()
        assistant = match.group(2).strip()

        if not user or not assistant:
            skipped += 1
            continue

        record = {
            "messages": [
                {"role": "user", "content": user},
                {"role": "assistant", "content": assistant}
            ]
        }

        out.write(json.dumps(record, ensure_ascii=False) + "\n")
        kept += 1

print("✅ Conversion complete")
print(f"✔ Kept samples  : {kept}")
print(f"⚠ Skipped rows : {skipped}")
print(f"📁 Output file : {OUTPUT_FILE}")


✅ Conversion complete
✔ Kept samples  : 1530
⚠ Skipped rows : 0
📁 Output file : swayam_phi3_clean.jsonl


In [ ]:
!pip install -U unsloth

In [ ]:
#Upload jsonl dataset
from google.colab import files
files.upload()

Saving swayam_phi3_dataset.jsonl to swayam_phi3_dataset.jsonl


{'swayam_phi3_dataset.jsonl': b'{"messages": [{"role": "user", "content": "Who are you?"}, {"role": "assistant", "content": "I am Swayam Satpathy, a Data Science and AI Engineer specializing in Machine Learning, Deep Learning, NLP, Computer Vision, and Generative AI."}]}\n{"messages": [{"role": "user", "content": "Can you briefly explain Who are you?"}, {"role": "assistant", "content": "I am Swayam Satpathy, a Data Science and AI Engineer specializing in Machine Learning, Deep Learning, NLP, Computer Vision, and Generative AI."}]}\n{"messages": [{"role": "user", "content": "How would you describe Who are you to a recruiter?"}, {"role": "assistant", "content": "I am Swayam Satpathy, a Data Science and AI Engineer specializing in Machine Learning, Deep Learning, NLP, Computer Vision, and Generative AI."}]}\n{"messages": [{"role": "user", "content": "Could you summarize Who are you?"}, {"role": "assistant", "content": "I am Swayam Satpathy, a Data Science and AI Engineer specializing in M

In [ ]:
!mkdir -p dataset
!mv *.jsonl dataset/

In [ ]:
!wc -l dataset/swayam_phi3_dataset.jsonl

wc: dataset/swayam_phi3_dataset.jsonl: No such file or directory


In [ ]:
#LOAD PHI-3 MINI (4-BIT, FAST)
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

max_seq_length = 4096
dtype = torch.float16
load_in_4bit = True   # QLoRA

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)


==((====))==  Unsloth 2026.1.4: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
#APPLY LoRA (OPTIMIZED FOR PHI-3)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0.05,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias = "none",
    use_gradient_checkpointing = True,  # BIG memory + speed win
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
#LOAD DATASET (JSONL CHAT FORMAT)
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files = "dataset/swayam_phi3_dataset.jsonl",
    split = "train"
)

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
#Formatting function
def format_phi3_chat(example):
    """
    Robust formatter for Phi-3 + Unsloth.
    Handles:
    - single example
    - batched examples
    - nested message lists
    MUST return List[str]
    """

    messages = example["messages"]

    outputs = []

    # Case 1: batched (list of conversations)
    if isinstance(messages[0], list):
        conversations = messages
    else:
        conversations = [messages]

    for convo in conversations:
        if not isinstance(convo, list) or len(convo) < 2:
            continue

        user_msg = None
        assistant_msg = None

        for msg in convo:
            if not isinstance(msg, dict):
                continue

            role = msg.get("role")
            content = msg.get("content")

            if role == "user" and isinstance(content, str):
                user_msg = content.strip()
            elif role == "assistant" and isinstance(content, str):
                assistant_msg = content.strip()

        if user_msg and assistant_msg:
            text = (
                f"<|user|>\n{user_msg}\n"
                f"<|assistant|>\n{assistant_msg}"
            )
            outputs.append(text)

    return outputs




In [ ]:
out = format_phi3_chat(dataset[0])
print(out)
print(type(out), len(out))


['<|user|>\nWho are you?\n<|assistant|>\nI am Swayam Satpathy, a Data Science and AI Engineer specializing in Machine Learning, Deep Learning, NLP, Computer Vision, and Generative AI.']
<class 'list'> 1


In [ ]:
#TRANING
from transformers import TrainingArguments
from trl import SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    formatting_func = format_phi3_chat,
    max_seq_length = 4096,
    packing = True,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        warmup_ratio = 0.05,
        fp16 = True,
        logging_steps = 25,
        optim = "adamw_8bit",
        lr_scheduler_type = "cosine",
        save_strategy = "epoch",
        output_dir = "/content/phi3-unsloth-lora",
        report_to = "none",
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1530 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,530 | Num Epochs = 3 | Total steps = 576
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,884,416 of 3,850,963,968 (0.78% trained)


Step,Training Loss
25,2.651500
50,1.297000
75,0.825100
100,0.537100
125,0.378800
150,0.288000
175,0.261100
200,0.235900
225,0.217600
250,0.217800


TrainOutput(global_step=576, training_loss=0.40350584419340724, metrics={'train_runtime': 1147.0848, 'train_samples_per_second': 4.001, 'train_steps_per_second': 0.502, 'total_flos': 5046147618140160.0, 'train_loss': 0.40350584419340724, 'epoch': 3.0})

In [ ]:
model.save_pretrained("/content/swayam_sat-phi3-unsloth-lora")
tokenizer.save_pretrained("/content/swayam_sat-phi3-unsloth-lora")


('/content/swayam_sat-phi3-unsloth-lora/tokenizer_config.json',
 '/content/swayam_sat-phi3-unsloth-lora/special_tokens_map.json',
 '/content/swayam_sat-phi3-unsloth-lora/chat_template.jinja',
 '/content/swayam_sat-phi3-unsloth-lora/tokenizer.model',
 '/content/swayam_sat-phi3-unsloth-lora/added_tokens.json',
 '/content/swayam_sat-phi3-unsloth-lora/tokenizer.json')

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/swayam_sat-phi3-unsloth-lora", "zip", "/content/swayam_sat-phi3-unsloth-lora")
files.download("/content/swayam_sat-phi3-unsloth-lora.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from unsloth import FastLanguageModel
from peft import PeftModel
import torch

BASE_MODEL = "microsoft/Phi-3-mini-4k-instruct"
ADAPTER_PATH = "/content/swayam_sat-phi3-unsloth-lora"

model, tokenizer = FastLanguageModel.from_pretrained(
    BASE_MODEL,
    max_seq_length=4096,
    dtype=torch.float16,
    load_in_4bit=True,
)

model = PeftModel.from_pretrained(model, ADAPTER_PATH)
FastLanguageModel.for_inference(model)


==((====))==  Unsloth 2026.1.4: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32009)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
 

In [ ]:
def chat(prompt, max_new_tokens=200):
    text = f"<|user|>\n{prompt}\n<|assistant|>"
    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
print(chat("What is your name"))

What is your name
 My name is Swayam Satpathy.



In [ ]:
print(chat("What are your goals"))

What are your goals
 My goals are to continuously improve as an AI engineer, work on impactful real-world problems, and stay updated with the latest AI research and trends.



In [ ]:
print(chat("Where are you based from?"))

Where are you based from?
 I am based in Berhampur, Odisha, India.



In [ ]:
print(chat("Give me a formal professional intro of yourself"))

Give me a formal professional intro of yourself
 I am a results-oriented AI Engineer with hands-on experience in building end-to-end AI systems using TensorFlow, PyTorch, and Streamlit, and deploying scalable solutions with Docker and cloud platforms.



In [ ]:
print(chat("What are your techinical skills?"))

What are your techinical skills?
 My technical skills include Python programming, machine learning, deep learning, NLP, generative AI, and building end-to-end AI systems using modern frameworks and tools.



In [ ]:
print(chat("What are you currently learning?"))

What are you currently learning?
 I am currently learning Python, SQL, machine learning, deep learning, NLP, and how to build end-to-end AI systems using modern frameworks and tools.



In [ ]:
!pip install -U unsloth transformers peft accelerate sentencepiece

  Using cached transformers-5.0.0-py3-none-any.whl.metadata (37 kB)


In [ ]:
from google.colab import files
import zipfile
import os

# This will prompt you to upload your zipped folder
# If you've already uploaded it, you can comment this line out.
files.upload()

# Assume the uploaded zip file is named 'my_folder.zip'
# You need to replace 'my_folder.zip' with the actual name of your uploaded zip file.
zip_file_name = "swayam_sat-phi3-unsloth-lora.zip"

if os.path.exists(zip_file_name):
    with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
        zip_ref.extractall(".")
    print(f"✅ Successfully unzipped {zip_file_name}")
    # Optional: remove the zip file after extraction
    # os.remove(zip_file_name)
else:
    print(f"❌ Error: {zip_file_name} not found. Please ensure it's uploaded.")


KeyboardInterrupt: 

In [ ]:
from unsloth import FastLanguageModel
from peft import PeftModel
import torch

BASE_MODEL = "microsoft/Phi-3-mini-4k-instruct"
ADAPTER_PATH = "/content/swayam_sat-phi3-unsloth-lora"
MERGED_PATH = "/content/swayam_sat-phi3-merged"

model, tokenizer = FastLanguageModel.from_pretrained(
    BASE_MODEL,
    max_seq_length=4096,
    dtype=torch.float16,
    load_in_4bit=False,   # 🚨 IMPORTANT
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.4: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
model = PeftModel.from_pretrained(model, ADAPTER_PATH)

# Merge LoRA weights into base model
model = model.merge_and_unload()

# Save merged model
model.save_pretrained(MERGED_PATH, safe_serialization=True)
tokenizer.save_pretrained(MERGED_PATH)

print("✅ Phi-3 Mini merged successfully")


✅ Phi-3 Mini merged successfully


In [ ]:
!git clone https://github.com/ggerganov/llama.cpp
!cd llama.cpp
!pip install -r requirements.txt


Cloning into 'llama.cpp'...
remote: Enumerating objects: 77188, done.
remote: Counting objects: 100% (364/364), done.
remote: Compressing objects: 100% (203/203), done.
remote: Total 77188 (delta 290), reused 161 (delta 161), pack-reused 76824 (from 3)
Receiving objects: 100% (77188/77188), 283.97 MiB | 26.96 MiB/s, done.
Resolving deltas: 100% (55784/55784), done.
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [ ]:
!cd /content/llama.cpp

In [ ]:
ls

dataset/                       swayam_dataset_ext.json
huggingface_tokenizers_cache/  swayam_sat-phi3-merged/
llama.cpp/                     swayam_sat-phi3-unsloth-lora/
phi3-unsloth-lora/             swayam_sat-phi3-unsloth-lora.zip
sample_data/                   unsloth_compiled_cache/


In [ ]:
!cd /content
!rm -rf llama.cpp
!git clone https://github.com/ggerganov/llama.cpp
!cd llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 77188, done.
remote: Counting objects: 100% (362/362), done.
remote: Compressing objects: 100% (204/204), done.
remote: Total 77188 (delta 290), reused 158 (delta 158), pack-reused 76826 (from 3)
Receiving objects: 100% (77188/77188), 283.87 MiB | 36.66 MiB/s, done.
Resolving deltas: 100% (55758/55758), done.
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [ ]:
!pip install -r /content/llama.cpp/requirements.txt

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 78.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.2/96.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!ls | grep gguf

In [ ]:
!python /content/llama.cpp/convert_hf_to_gguf.py \
  /content/swayam_sat-phi3-merged \
  --outfile /content/swayam-phi3-f16.gguf \
  --outtype f16

INFO:hf-to-gguf:Loading model: swayam_sat-phi3-merged
INFO:hf-to-gguf:Model architecture: MistralForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00002.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00002.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {3072, 32064}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {8192, 3072}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.attn_

In [ ]:
ls -lh /content/swayam-phi3-f16.gguf

-rw-r--r-- 1 root root 7.2G Jan 26 20:19 /content/swayam-phi3-f16.gguf


In [ ]:
!cd /content/llama.cpp
!make quantize

make: *** No rule to make target 'quantize'.  Stop.


In [ ]:
ls /content/llama.cpp/quantize

ls: cannot access '/content/llama.cpp/quantize': No such file or directory
